In [ ]:
import numpy as np
import time
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [4]:
# Hyperparameters
n_pca_components = 500
n_tsne_components = 3
perplexity = 500
random_state = 42

runs_to_process = range(1, 26)
total_runs = len(runs_to_process)

# Actuation Strouhal numbers for coloring later
st_info = {
    1: 0.0, 2: 0.05, 3: 0.10, 4: 0.15, 5: 0.20, 6: 0.25, 7: 0.30, 8: 0.35, 9: 0.40, 
    10: 0.45, 11: 0.50, 12: 0.55, 13: 0.60, 14: 0.65, 15: 0.70, 16: 0.75, 17: 0.80, 
    18: 0.85, 19: 0.90, 20: 0.95, 21: 1.00, 22: 1.05, 23: 1.10, 24: 1.15, 25: 1.20
}

output_path = "tsne_data/global_tsne_3D_perp_500.npz"


data = np.load('pca_data/global_forced_pca.npz')
global_pca_scores   = data['pca_scores']    # (48720, 500)
run_labels   = data['run_labels']    # (48720,)
st_labels    = data['st_labels']     # (48720,)
phase_labels = data['phase_labels']  # (48720,)


# ---------------------------------------------------------
# Compute t-SNE
# ---------------------------------------------------------
print(f"\n--- Computing 3D t-SNE (perplexity={perplexity}) ---")
start_time = time.time()

tsne = TSNE(n_components=n_tsne_components, perplexity=perplexity, 
            learning_rate='auto', init='pca', random_state=random_state, n_jobs=-1)

X_tsne = tsne.fit_transform(X_global_pca)

computation_time = time.time() - start_time
print(f"\nComputation finished in {computation_time/60:.1f} minutes.")

print(f"Saving results to {output_path}...")
np.savez_compressed(
    output_path,
    embedding=X_tsne,
    run_labels=run_labels,
    st_labels=st_labels,
    perplexity=perplexity,
    n_pca_components=n_pca_components,
    time_sec=computation_time
)


--- PASS 3: Computing 3D t-SNE (perplexity=500) ---


KeyboardInterrupt: 

In [ ]:
# Load the computed t-SNE
data = np.load("tsne_data/global_tsne_3D_perp_500.npz")
emb = data['embedding']
st_labels = data['st_labels']
run_labels = data['run_labels']

# We can plot all points, but setting alpha helps see overlapping structures
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot colored by Strouhal number
# Using a colormap (e.g. viridis or jet) to represent St
p = ax.scatter(emb[:, 0], emb[:, 1], emb[:, 2], 
               c=st_labels, cmap='jet', s=2, alpha=0.6, edgecolors='none')

cbar = plt.colorbar(p, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Actuation Strouhal Number ($St_{act}$)', fontsize=12)

ax.set_title("Global 3D t-SNE Manifold of All Runs", fontsize=15)
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.set_zlabel("t-SNE Dimension 3")

plt.tight_layout()
plt.savefig('figures/global_tsne_3D.png', dpi=300, bbox_inches='tight')
plt.show()

# Print PCA Variance info
if 'var_explained' in data:
    print(f"Variance explained by PCA step: {data['var_explained']:.2%}")
